# Provision Block Storage — proj03

Creates a **20 GiB Cinder block volume** on **KVM@TACC** for persisting small application state (PostgreSQL, RabbitMQ, etc.) across VM teardowns.

**Run this ONCE.** The volume survives VM teardowns — do not delete it unless decommissioning the project.

**Resources created:**
- Cinder volume: `block-<username>-proj03` (20 GiB, KVM@TACC)

**Next step:** Run `provision_vm.ipynb` to create the VM and attach this volume.

In [ ]:
from chi import context
import chi, os, time

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")
username = os.getenv("USER")
project = "proj03"

print(f"User: {username} | Project: {project} | Site: KVM@TACC")

In [ ]:
cinder_client = chi.clients.cinder()
volume_name = f"block-{username}-{project}"

# Idempotent — skip creation if volume already exists
existing = [v for v in cinder_client.volumes.list() if v.name == volume_name]
if existing:
    volume = existing[0]
    print(f"Volume '{volume_name}' already exists (status: {volume.status}) — skipping creation.")
else:
    volume = cinder_client.volumes.create(name=volume_name, size=20)
    print(f"Volume '{volume_name}' created (20 GiB). Waiting for AVAILABLE...")
    while True:
        volume = cinder_client.volumes.get(volume.id)
        if volume.status == "available":
            break
        time.sleep(3)
    print(f"Volume '{volume_name}' is AVAILABLE.")

print(f"\nVolume ID: {volume.id}")
print("Run provision_vm.ipynb next — it will attach this volume to the server.")

In [ ]:
# ---------------------------------------------------------------------------
# Extend volume (optional — skip if no resize is needed)
# ---------------------------------------------------------------------------

NEW_SIZE_GIB = 40  # set to desired GiB — must be larger than current size

os_conn = chi.clients.connection()
volume = cinder_client.volumes.get(volume.id)  # refresh

if NEW_SIZE_GIB > volume.size:
    print(f"Extending volume from {volume.size} GiB to {NEW_SIZE_GIB} GiB...")

    # The volume's stored volume_type_id may reference a deleted type, which
    # causes os-extend to 404. Retype to a valid type first to fix the metadata.
    valid_types = list(os_conn.block_storage.types())
    if not valid_types:
        raise RuntimeError("No volume types available on this site.")
    target_type = next((t for t in valid_types if t.name == "__DEFAULT__"), valid_types[0])
    print(f"Retyping volume to '{target_type.name}' to fix stale volume_type_id...")
    os_conn.block_storage.retype_volume(
        volume.id, new_type=target_type.name, migration_policy="never"
    )

    # Wait for retype to settle
    for _ in range(20):
        vol = os_conn.block_storage.get_volume(volume.id)
        if vol.status == "available":
            break
        time.sleep(3)

    os_conn.block_storage.extend_volume(volume.id, NEW_SIZE_GIB)
    while True:
        vol = os_conn.block_storage.get_volume(volume.id)
        if vol.status == "available":
            break
        elif vol.status == "error":
            raise RuntimeError("Volume extend failed — volume entered 'error' state.")
        time.sleep(3)
    print(f"Volume extended to {NEW_SIZE_GIB} GiB.")
    # NOTE: if the volume is attached to a VM, also run inside the VM:
    #       sudo resize2fs /dev/vdb1
else:
    print(f"Volume already at {volume.size} GiB — no resize needed.")